# Notebook 4 — Model Training

Trains XGBoost, CNN, and probability-averaging ensembles for Signature, GARCH, and Combined feature sets across all selected experiments.

In [6]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

In [7]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
FEATURE_ROOT = Path("../data/features")
MODEL_ROOT = Path("../models")
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

In [8]:
RUN_EXPERIMENTS = [
    "monthly_ff3",
    "monthly_ff5",
    "daily_ff3",
    "daily_ff5",
]

EXPERIMENT_CONFIG = {
    "monthly_ff3": {"frequency": "monthly", "factor_model": "ff3", "window": 12, "periods_per_year": 12},
    "monthly_ff5": {"frequency": "monthly", "factor_model": "ff5", "window": 12, "periods_per_year": 12},
    "daily_ff3": {"frequency": "daily", "factor_model": "ff3", "window": 60, "periods_per_year": 252},
    "daily_ff5": {"frequency": "daily", "factor_model": "ff5", "window": 60, "periods_per_year": 252},
}

unknown = set(RUN_EXPERIMENTS) - set(EXPERIMENT_CONFIG)
if unknown:
    raise ValueError(f"Unknown experiments: {sorted(unknown)}")

In [9]:
def evaluate(y_true, prob, threshold, name):
    pred = (prob >= threshold).astype(int)
    return {
        "model": name, "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, prob),
    }

def find_best_threshold(y_true, prob):
    rows=[]
    for t in np.arange(0.05, 0.96, 0.01):
        pred=(prob>=t).astype(int)
        rows.append((t, f1_score(y_true,pred,zero_division=0)))
    return max(rows, key=lambda x:x[1])[0]

def train_xgb(X_train, y_train, X_val, y_val, scale_pos_weight):
    model=XGBClassifier(
        objective="binary:logistic", eval_metric="logloss", n_estimators=600,
        learning_rate=0.03, max_depth=4, min_child_weight=3, subsample=0.8,
        colsample_bytree=0.8, reg_lambda=1.0, scale_pos_weight=scale_pos_weight,
        random_state=SEED, n_jobs=-1
    )
    model.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=False)
    return model

def reshape_for_cnn(X): return X.reshape(X.shape[0], X.shape[1], 1)

def build_cnn(input_shape):
    model=models.Sequential([
        layers.Input(shape=input_shape), layers.Conv1D(32,3,padding="same",activation="relu"),
        layers.BatchNormalization(), layers.Conv1D(64,3,padding="same",activation="relu"),
        layers.BatchNormalization(), layers.GlobalAveragePooling1D(), layers.Dense(64,activation="relu"),
        layers.Dropout(0.30), layers.Dense(32,activation="relu"), layers.Dropout(0.20),
        layers.Dense(1,activation="sigmoid")])
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss="binary_crossentropy",
                  metrics=[tf.keras.metrics.AUC(name="auc")])
    return model

def find_best_ensemble(y, cnn_prob, xgb_prob):
    best=None
    for w in np.arange(0,1.01,0.05):
        p=w*cnn_prob+(1-w)*xgb_prob
        t=find_best_threshold(y,p)
        score=f1_score(y,(p>=t).astype(int),zero_division=0)
        row={"cnn_weight":float(w),"xgb_weight":float(1-w),"threshold":float(t),"f1":float(score)}
        if best is None or row["f1"]>best["f1"]: best=row
    return best

In [11]:
for experiment in RUN_EXPERIMENTS:
    print()
    print("=" * 80)
    print(experiment)
    print("=" * 80)

    feature_dir=FEATURE_ROOT/experiment
    model_dir=MODEL_ROOT/experiment
    model_dir.mkdir(parents=True,exist_ok=True)
    y_train=np.load(feature_dir/"y_train.npy")
    y_val=np.load(feature_dir/"y_validation.npy")
    y_test=np.load(feature_dir/"y_test.npy")
    scale_pos_weight=(y_train==0).sum()/max((y_train==1).sum(),1)

    feature_sets={}
    for feature in ["signature","garch","combined"]:
        feature_sets[feature]={s:np.load(feature_dir/f"X_{feature}_{s}.npy") for s in ["train","validation","test"]}

    xgb_models={}; cnn_models={}; val_probs={}; test_probs={}; thresholds={}; histories={}; results=[]; ensemble_settings={}
    class_weight={0:1.0,1:float(scale_pos_weight)}

    for feature,d in feature_sets.items():
        xgb=train_xgb(d["train"],y_train,d["validation"],y_val,scale_pos_weight)
        xgb_val=xgb.predict_proba(d["validation"])[:,1]
        xgb_test=xgb.predict_proba(d["test"])[:,1]
        xgb_t=find_best_threshold(y_val,xgb_val)
        results.append(evaluate(y_test,xgb_test,xgb_t,f"XGBoost {feature.title()} — Test"))
        xgb.save_model(model_dir/f"xgb_{feature}.json")
        xgb_models[feature]=xgb
        val_probs[("xgb",feature)]=xgb_val; test_probs[("xgb",feature)]=xgb_test; thresholds[("xgb",feature)]=xgb_t

        Xtr,Xv,Xte=map(reshape_for_cnn,[d["train"],d["validation"],d["test"]])
        cnn=build_cnn(Xtr.shape[1:])
        history=cnn.fit(Xtr,y_train,validation_data=(Xv,y_val),epochs=100,batch_size=64,
            class_weight=class_weight,verbose=0,callbacks=[
                callbacks.EarlyStopping(monitor="val_auc",mode="max",patience=12,restore_best_weights=True),
                callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=5,min_lr=1e-6)])
        cnn_val=cnn.predict(Xv,verbose=0).ravel(); cnn_test=cnn.predict(Xte,verbose=0).ravel()
        cnn_t=find_best_threshold(y_val,cnn_val)
        results.append(evaluate(y_test,cnn_test,cnn_t,f"CNN {feature.title()} — Test"))
        cnn.save(model_dir/f"cnn_{feature}.keras")
        pd.DataFrame(history.history).to_csv(model_dir/f"cnn_{feature}_history.csv",index=False)
        cnn_models[feature]=cnn
        val_probs[("cnn",feature)]=cnn_val; test_probs[("cnn",feature)]=cnn_test; thresholds[("cnn",feature)]=cnn_t

        setting=find_best_ensemble(y_val,cnn_val,xgb_val)
        ensemble_settings[feature]=setting
        ensemble_test=setting["cnn_weight"]*cnn_test+setting["xgb_weight"]*xgb_test
        results.append(evaluate(y_test,ensemble_test,setting["threshold"],f"Ensemble {feature.title()} — Test"))
        test_probs[("ensemble",feature)]=ensemble_test

    pd.DataFrame(results).sort_values(["f1","roc_auc"],ascending=False).to_csv(model_dir/"final_model_results.csv",index=False)
    pd.DataFrame([{"feature_set":f,**s} for f,s in ensemble_settings.items()]).to_csv(model_dir/"ensemble_settings.csv",index=False)
    pd.DataFrame([{"feature_set":f,"threshold":thresholds[("xgb",f)]} for f in feature_sets]).to_csv(model_dir/"xgb_thresholds.csv",index=False)
    pd.DataFrame([{"feature_set":f,"threshold":thresholds[("cnn",f)]} for f in feature_sets]).to_csv(model_dir/"cnn_thresholds.csv",index=False)
    pred_df=pd.DataFrame({"actual_label":y_test})
    for architecture in ["xgb","cnn","ensemble"]:
        for feature in feature_sets:
            pred_df[f"{architecture}_{feature}_prob"]=test_probs[(architecture,feature)]
    pred_df.to_csv(model_dir/"test_probabilities.csv",index=False)
    print("Saved models and results to",model_dir.resolve())


monthly_ff3
Saved models and results to C:\Users\kyler\Documents\VS_Code\Finance Code\ML using FAMA and FRENCH\models\monthly_ff3

monthly_ff5
Saved models and results to C:\Users\kyler\Documents\VS_Code\Finance Code\ML using FAMA and FRENCH\models\monthly_ff5

daily_ff3
Saved models and results to C:\Users\kyler\Documents\VS_Code\Finance Code\ML using FAMA and FRENCH\models\daily_ff3

daily_ff5
Saved models and results to C:\Users\kyler\Documents\VS_Code\Finance Code\ML using FAMA and FRENCH\models\daily_ff5
